In [0]:
# Imports
# We introduce two new imports here:
# - round as spark_round: rounds decimal values (aliased to avoid
#   shadowing Python's built-in round() function)
# - date_format: formats timestamps into readable strings
# - DeltaTable: the Python class that gives us MERGE INTO capability

from pyspark.sql.functions import col, round as spark_round, current_timestamp, date_format
from delta.tables import DeltaTable

print("Imports successful")

In [0]:
# Reading Bronze GDP and dim_country
# These are the two inputs to this Silver notebook.
# Bronze GDP gives us raw economic data.
# dim_country gives us the validated country reference list.
# We read both using spark.table() — the Unity Catalog pattern.
# No transformation yet. Read first, transform second

df_bronze_gdp = spark.table("workspace.default.bronze_world_bank_gdp")
df_dim_country = spark.table("workspace.default.dim_country")

print(f"Bronze GDP rows: {df_bronze_gdp.count()}")
print(f"dim_country rows: {df_dim_country.count()}")

# Bronze data shape
print("\nBronze GDP schema:")
df_bronze_gdp.printSchema()

In [0]:
display(df_bronze_gdp)

In [0]:
display(df_dim_country)

In [0]:
# Transform: select, round, join
# Three things happen in this single step:
#
# 1. SELECT — we keep only the columns Silver needs.
#    Bronze had 10 columns. Silver needs 5 from Bronze.
#    Keeping unnecessary columns wastes storage and confuses
#    downstream consumers.
#
# 2. ROUND — gdp_per_capita_usd rounded to 2 decimal places.
#    Bronze stores raw API precision (12 decimal places).
#    Silver stores human-readable precision.
#    spark_round(col, 2) is the PySpark way to do this.
#
# 3. JOIN — inner join against dim_country on country_code.
#    This does two things simultaneously:
#    a) Validates: only rows with a recognised country_code survive
#    b) Enriches: brings in region and country_iso3 from dim_country
#
# Why inner join specifically?
# If Bronze somehow has a typo'd country code not in dim_country,
# that row gets dropped here rather than flowing downstream as bad data.
# dim_country is our source of truth. Inner join enforces that.

df_silver_gdp = (
    df_bronze_gdp
    .select(
        col("country_code"),
        col("year"),
        spark_round(col("value"), 2).alias("gdp_per_capita_usd")
    )
    .join(
        df_dim_country.select("country_code", "country_iso3", "country_name", "region"),
        on="country_code",
        how="inner"
    )
    .select(
        "country_code",
        "country_iso3",
        "country_name",
        "region",
        "year",
        "gdp_per_capita_usd"
    )
)

# Add Silver metadata — when this record was last processed
df_silver_gdp = df_silver_gdp.withColumn(
    "updated_at",
    date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
)

print(f"Silver GDP rows: {df_silver_gdp.count()}")
display(df_silver_gdp.orderBy("country_code", "year").limit(10))

In [0]:
# First-time write to Silver
# We use overwrite here because this is the FIRST time we are
# creating this Silver table. The table does not exist yet.
# overwrite creates it cleanly with the correct schema.
#
# IMPORTANT: This cell should only ever run ONCE.
# After this cell runs successfully, all future pipeline runs
# will use MERGE INTO instead of this cell.
# Running this cell again after MERGE records exist would
# wipe out any update history — which defeats the purpose.
#
# Think of this cell as "laying the foundation".
# MERGE INTO is "adding floors to the building".
# You only lay the foundation once.

TABLE_NAME = "workspace.default.silver_fact_gdp"

(df_silver_gdp.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Silver table created: {TABLE_NAME}")
print(f"Row count: {spark.table(TABLE_NAME).count()}")

In [0]:
# MERGE INTO (the heart of Silver)
# The merge works in three parts:
#
# PART 1: deltaTable — the TARGET
# This is the existing Silver table we want to update.
# DeltaTable.forName() loads it by Unity Catalog name.
#
# PART 2: df_silver_gdp — the SOURCE
# This is the fresh DataFrame we just transformed from Bronze.
# It contains the latest values from the API.
#
# PART 3: The merge condition
# "target.country_code = source.country_code AND
#  target.year = source.year"
# This is how Spark identifies matching rows.
# country_code + year together form the NATURAL KEY —
# the unique identifier for one GDP observation.
# No two rows should ever share the same country + year.
#
# WHEN MATCHED → UPDATE
# The row already exists. Overwrite its values with
# the latest data. This handles data corrections —
# if the World Bank revises a historical figure,
# the next pipeline run automatically fixes it.
#
# WHEN NOT MATCHED → INSERT
# This is a new country or a new year we haven't seen before.
# Add it as a fresh row.

TABLE_NAME = "workspace.default.silver_fact_gdp"

# Load the existing Silver table as a DeltaTable object
# DeltaTable is Delta Lake's Python class for write operations
# spark.table() gives you a DataFrame (read-only)
# DeltaTable.forName() gives you a DeltaTable (read + write + merge)
delta_target = DeltaTable.forName(spark, TABLE_NAME)

(
    delta_target.alias("target")
    .merge(
        df_silver_gdp.alias("source"),
        "target.country_code = source.country_code AND target.year = source.year"
    )
    .whenMatchedUpdate(set={
        "gdp_per_capita_usd": "source.gdp_per_capita_usd",
        "country_iso3":       "source.country_iso3",
        "country_name":       "source.country_name",
        "region":             "source.region",
        "updated_at":         "source.updated_at",
    })
    .whenNotMatchedInsert(values={
        "country_code":       "source.country_code",
        "country_iso3":       "source.country_iso3",
        "country_name":       "source.country_name",
        "region":             "source.region",
        "year":               "source.year",
        "gdp_per_capita_usd": "source.gdp_per_capita_usd",
        "updated_at":         "source.updated_at",
    })
    .execute()
)

print("MERGE complete")
print(f"Row count after MERGE: {spark.table(TABLE_NAME).count()}")

In [0]:
# Verify Silver GDP table
# Three checks in one cell:
#
# CHECK 1: Row count per country
# Every country should have exactly 14 rows (2010-2023)
# If any country has fewer, it means some years had null GDP
# values in Bronze which got skipped during flattening.
# That is acceptable — not every country reports every year.
#
# CHECK 2: GDP value ranges
# We verify the rounded values look realistic.
# Sub-Saharan African GDP per capita typically ranges
# from ~$300 (very poor economies) to ~$7,000.
# If we see values outside this range something is wrong.
#
# CHECK 3: Confirm region column came through from dim_country join
# This proves the join worked and dim_country is enriching our data.

df_check = spark.table("workspace.default.silver_fact_gdp")

print(f"Total rows: {df_check.count()}")

print("\nRows per country (should be 14 per country for 2010-2023):")
display(
    df_check.groupBy("country_code", "country_name", "region")
            .count()
            .orderBy("region", "country_name")
)

In [0]:
# GDP value range check
from pyspark.sql.functions import avg, min, max

print("GDP per capita ranges (USD):")
display(
    df_check.groupBy("country_code", "country_name")
            .agg(
                spark_round(min("gdp_per_capita_usd"),  2).alias("min_gdp"),
                spark_round(avg("gdp_per_capita_usd"),  2).alias("avg_gdp"),
                spark_round(max("gdp_per_capita_usd"),  2).alias("max_gdp"),
            )
            .orderBy("avg_gdp", ascending=False)
)

In [0]:
# Kenya GDP trend
# Kenya's GDP should show a growth trend from 2010 to 2023
# with a visible dip around 2020 (COVID economic impact).

print("Kenya GDP per capita trend 2010-2023:")
display(
    df_check.filter(col("country_code") == "KE")
            .select("year", "country_name", "gdp_per_capita_usd")
            .orderBy("year")
)